<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 09 · 换一种说法，还能找到吗

记忆里写的是“坏行要带原始 CSV 行号”，用户问的却是“怎样告诉运营人员哪条数据有问题”。
如果两边没有共享关键词，单纯的文字匹配可能不够。语义检索为这个问题提供了另一条路径。

我们会使用同一批真实存储的 Memory，对比 fts、vector 和 hybrid。先看命中内容，再看排名和耗时，
让结果说明这些方法在当前数据与模型下各自适合什么问题。

**这一篇的收获：** 配置 Embedding，实际比较三种检索模式，检查 matched_by，并用小型标注集计算可解释的命中指标。

**运行准备：** 从 [教程入口](README.md) 安装依赖并启动 Jupyter。每篇都带有自己的数据，可以独立运行。
本篇调用真实模型，请先完成 README 的模型配置；调用会产生所选服务的用量。
建议先逐格运行，读完输出再继续；完整重跑时使用 **Restart Kernel & Run All**。

## 先准备一个自己的实验空间

下面的辅助代码只负责启动本地 Server、建立 Client 和整理输出。默认每次完整运行使用新的 SQLite 数据库；选择 OceanBase 时，使用专用测试库并为本次实验创建新的 Scope。
后端设置见 [README](README.md#使用-oceanbase-运行)。关键的写入、检索、审核与交接调用会直接写在后面的单元格里。


本篇通过 `remember_memory` 写入 Scope 的日常 Memory，使后续搜索和上下文准备读取同一份知识。独立制品的创建、版本管理与日常记忆的关系见 [接口选择](DESIGN.md#接口选择)。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))

if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("09", features=("embedding",))
client = lab.client
assert client is not None

现在创建本篇的项目 Scope。`title` 是给人看的名称，真正用于调用的是 Server 返回的 `scope_id`。
你可以改变标题；不要自己根据标题或目录拼出一个 Scope ID。


In [ ]:
from powercontext.http import CreateScopeRequest

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 09",
        summary="第 09 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-09",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 检查可用模式

向量检索需要模型、稳定的 profile ID 和正确的输出维度。开头的实验环境只启用 embedding，
不会调用生成模型。所有 Memory 都由我们显式写入，这样检索对比不会混入提取差异。


In [ ]:
from powercontext.http import RememberMemoryRequest, SearchMemoryRequest

capabilities = await client.get_capabilities()
assert {"fts", "vector", "hybrid"} <= set(capabilities.search_modes)
show({"可用模式": capabilities.search_modes})

## 2. 保存一批可区分的约定

每条约定有一个简短主题标签。保存后记录实际 entry_id，后面用它判断是否找到了预期条目，
避免把正文里恰好出现某个词当成检索成功。


In [ ]:
memories = {
    "amount": "amount: Store order amounts as integer cents. Reject fractional cents rather than silently rounding them.",
    "line_number": "line_number: When a CSV row is invalid, report its original line number so operators can locate and repair the bad record.",
    "privacy": "privacy: Error messages must redact credentials and personal information. Never echo a complete raw CSV row.",
    "batch_limit": "batch_limit: Import at most 200 records per batch. Ask the operator to split larger files.",
    "currency": "currency: Domestic orders use CNY; overseas orders use USD. Read the current project convention.",
    "release": "release: Publish only after the release checklist is complete and the maintenance window is confirmed.",
    "documentation": "documentation: Explain each example's purpose and expected result using short paragraphs.",
}
entry_ids = {}
for topic, text in memories.items():
    result = await client.remember_memory(
        RememberMemoryRequest(scope_id=scope_id, kind="constraint", text=text, reason="检索实验数据")
    )
    assert result.entry is not None
    entry_ids[topic] = result.entry.citation.entry_id
table([{"主题": topic, "内容": text} for topic, text in memories.items()])

## 3. 同一个问题，三种检索

先用没有直接写出 `line_number` 的中文问题，去检索下面用英文写好的约定。这是刻意制造的词面缺口：
FTS 看不到共享关键词时出现「无命中」是预期现象，不是故障。vector 使用配置的 Embedding，hybrid 结合两类信号。
跨语言匹配质量取决于所选模型，不是所有模型都会给出相同的排序。

耗时是当前机器和服务上的一次观察，包含网络与模型开销；它不是基准测试结论。


In [ ]:
import time

query = "怎样告诉运营人员哪条导入数据有问题，方便他们找到并修复？"
rows = []
for mode in ["fts", "vector", "hybrid"]:
    started = time.perf_counter()
    response = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query=query, mode=mode, limit=3))
    elapsed = round((time.perf_counter() - started) * 1000)
    if not response.hits:
        rows.append({"模式": mode, "排名": "—", "内容": "无命中", "匹配来源": "—", "耗时 ms": elapsed})
    for rank, hit in enumerate(response.hits, 1):
        rows.append({
            "模式": mode,
            "排名": rank,
            "内容": hit.text,
            "匹配来源": ", ".join(hit.matched_by),
            "耗时 ms": elapsed,
        })
    if mode == "vector":
        assert response.mode == "vector" and response.hits
        assert all("vector" in hit.matched_by for hit in response.hits)
table(rows)

## 4. 用少量已标注的问题检查命中

下面三种输入分别是字段名、英文改写和中文语义。每个问题都明确标注一个目标条目，
指标只回答“目标是否在前三条”和“排在第几位”。这是教学用的三题小样本，不代表总体检索效果。

不要直接比较不同模式的 score 大小；更有意义的是看目标身份、排名和正文。


In [ ]:
questions = [
    ("精确字段名", "line_number", "line_number"),
    ("英文改写", "Help the operator locate the offending record in the original import file.", "line_number"),
    ("中文语义", "错误提示怎样避免泄漏用户资料和密钥？", "privacy"),
]
observations = []
for label, question, target in questions:
    for mode in ["fts", "vector", "hybrid"]:
        response = await client.search_memory(
            SearchMemoryRequest(scope_id=scope_id, query=question, mode=mode, limit=3)
        )
        rank = next((i for i, hit in enumerate(response.hits, 1) if hit.citation.entry_id == entry_ids[target]), None)
        observations.append({
            "问题类型": label,
            "模式": mode,
            "目标": target,
            "Top-3 命中": rank is not None,
            "目标排名": rank,
        })
table(observations)
table([
    {"模式": mode, "Hit@3": sum(row["Top-3 命中"] for row in observations if row["模式"] == mode) / len(questions)}
    for mode in ["fts", "vector", "hybrid"]
])
assert any(row["问题类型"] == "精确字段名" and row["模式"] == "fts" and row["Top-3 命中"] for row in observations)

## 轮到你：增加一个容易混淆的约定

把“日志行号”和“CSV 原始行号”作为两个主题，让问题更接近真实项目。先写出你期望找到哪个条目，
再运行查询检查前三条。下面只提供一个可运行的起点，你可以更换提问和目标。
如果没有命中，先检查字段、语料和 Embedding 能力，再考虑是否需要调整检索配置。


In [ ]:
extra = await client.remember_memory(
    RememberMemoryRequest(
        scope_id=scope_id,
        kind="constraint",
        text="log_line: Diagnostic log line numbers identify application log entries, not rows in the uploaded CSV file.",
        reason="练习：加入邻近但不同的概念",
    )
)
my_query = "Find the invalid record in the uploaded CSV, not the diagnostic log."
my_result = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query=my_query, mode="hybrid", limit=3))
table([
    {"排名": rank, "预期条目": hit.citation.entry_id == entry_ids["line_number"], "内容": hit.text}
    for rank, hit in enumerate(my_result.hits, 1)
])

## 带着结果离开

你已经用真实 Embedding 验证了 vector 路径，并用相同数据比较检索结果。小型标注集让“找得好不好”成为可以继续检查和改进的问题。

下面关闭本篇的 Client 和 Server。实验文件仍留在教程的 `.powercontext/` 子目录，便于检查；
清理方法见 [README](README.md#清理实验数据)。如果在中途停止，请运行这个单元格，或关闭 Kernel。

下一篇：[给真实 Agent 接上 PowerContext](10_agent_memory.ipynb)。


## 补充实验：对召回候选再做一次排序

FTS、Vector、Hybrid 负责召回候选。Rerank 使用真实生成模型，在候选集合中选择更适合回答问题的条目，会增加耗时。

本单元格额外需要 Generation 模型。我们保留同一数据与问题，比较启用前后的排名，并检查服务统计中的模型调用；结果不必每次都变好，也不必每次都改变顺序。

In [ ]:
from _tutorial import inference_settings

from powercontext.builtin.runtime.config import RuntimeConfig
from powercontext.http import GetStatsRequest, ScopeSelection

query = "Find the invalid record in the uploaded CSV, not a diagnostic application log entry."
started = time.perf_counter()
coarse = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query=query, mode="hybrid", limit=3))
coarse_ms = round((time.perf_counter() - started) * 1000)
lab.inference = inference_settings(("embedding", "generation"))
lab.settings_overrides["runtime"] = RuntimeConfig(memory_rerank_enabled=True, memory_rerank_candidate_limit=8)
await lab.restart()
client = lab.client
selection = ScopeSelection.model_validate({"mode": "exact", "scope_ids": [scope_id]})
before_usage = await client.get_stats(GetStatsRequest(selection=selection))
started = time.perf_counter()
reranked = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query=query, mode="hybrid", limit=3))
rerank_ms = round((time.perf_counter() - started) * 1000)
after_usage = await client.get_stats(GetStatsRequest(selection=selection))
assert after_usage.usage.totals.generation.requests > before_usage.usage.totals.generation.requests
assert reranked.hits and all(
    hit.citation.entry_id in {*entry_ids.values(), extra.entry.citation.entry_id} for hit in reranked.hits
)
table([
    {"阶段": name, "排名": rank, "耗时 ms": elapsed, "内容": hit.text}
    for name, response, elapsed in (("Hybrid", coarse, coarse_ms), ("Hybrid + LLM rerank", reranked, rerank_ms))
    for rank, hit in enumerate(response.hits, 1)
])

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")